<a href="https://colab.research.google.com/github/umang0015/Ai-DataScience/blob/main/11AugSession2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/Models/train - train.csv')

In [ ]:
df.head()

,id,url_legal,license,excerpt,target,standard_error
0,c12129c31,NaN,NaN,When the young people returned to the ballroom...,-0.340259,0.464009
1,85aa80a4c,NaN,NaN,"All through dinner time, Mrs. Fayre was somewh...",-0.315372,0.480805
2,b69ac6792,NaN,NaN,"As Roger had predicted, the snow departed as q...",-0.580118,0.476676
3,dd1000b26,NaN,NaN,And outside before the palace a great garden w...,-1.054013,0.450007
4,37c1b32fb,NaN,NaN,Once upon a time there were Three Bears who li...,0.247197,0.510845


In [ ]:
df.isnull().sum()

,0
id,0
url_legal,2004
license,2004
excerpt,0
target,0
standard_error,0


In [ ]:
X = df['excerpt']
y = df['target']

In [ ]:
print(df.excerpt[0])
print(df.excerpt[2])

When the young people returned to the ballroom, it presented a decidedly changed appearance. Instead of an interior scene, it was a winter landscape.
The floor was covered with snow-white canvas, not laid on smoothly, but rumpled over bumps and hillocks, like a real snow field. The numerous palms and evergreens that had decorated the room, were powdered with flour and strewn with tufts of cotton, like snow. Also diamond dust had been lightly sprinkled on them, and glittering crystal icicles hung from the branches.
At each end of the room, on the wall, hung a beautiful bear-skin rug.
These rugs were for prizes, one for the girls and one for the boys. And this was the game.
The girls were gathered at one end of the room and the boys at the other, and one end was called the North Pole, and the other the South Pole. Each player was given a small flag which they were to plant on reaching the Pole.
This would have been an easy matter, but each traveller was obliged to wear snowshoes.
As Roge

In [ ]:
import nltk
import re
import regex
import numpy as np
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# ml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score , confusion_matrix, classification_report


In [ ]:
# downloading files which is used for nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
# creating the preprocessing pipeline
def text_process(text):
  import string # Moved import inside function for robust scoping
  # first lowering case
  text = text.lower()
  # removing the digits from text (re.sub(this is used to replace the substrings))
  text = re.sub(r'\d+' , '' , text)
  # removing punctuation from text
  text = text.translate(str.maketrans('','',string.punctuation))
  # remove the white space
  text  = text.strip()

  # now tokenize the text
  tokens = text.split()

  # now remove the stopwords
  final_tokens= [word for word in tokens if word not in stopwords.words("english")] # Corrected 'word' to 'words'

  ps = PorterStemmer()
  final_tokens = [ps.stem(word) for word in final_tokens]

  text = ' '.join(final_tokens)
  return text

In [ ]:
df['cleaned_message'] = df['excerpt'].apply(text_process)
df['cleaned_message']

,cleaned_message
0,young peopl return ballroom present decidedli ...
1,dinner time mr fayr somewhat silent eye rest d...
2,roger predict snow depart quickli came two day...
3,outsid palac great garden wall round fill full...
4,upon time three bear live togeth hous wood one...
...,...
2829,think dinosaur live pictur see hot steami swam...
2830,solid solid usual hard molecul pack togeth clo...
2831,second state matter discuss liquid solid hard ...
2832,solid shape actual touch three dimens mean len...


In [ ]:
df['target_class'] = pd.cut(
    df['target'],
    bins=[-float('inf'), -1.5, -0.5, float('inf')],
    labels=['Low', 'Medium', 'High']
)

In [ ]:
df[['target', 'target_class']].head(10)

,target,target_class
0,-0.340259,High
1,-0.315372,High
2,-0.580118,Medium
3,-1.054013,Medium
4,0.247197,High
5,-0.861809,Medium
6,-1.759061,Low
7,-0.952325,Medium
8,-0.371641,High
9,-1.238432,Medium


In [ ]:
df['target_class'].value_counts()

,count
target_class,
Medium,993
High,965
Low,876


In [ ]:
df['target_class'].value_counts(normalize=True) * 100

,proportion
target_class,
Medium,35.038814
High,34.050812
Low,30.910374


In [ ]:
X = df['cleaned_message'] # Changed from df['excerpt'] to df['cleaned_message']
y = df['target_class']

In [ ]:
from sklearn.model_selection import train_test_split
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2, random_state=42)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

Cbow = CountVectorizer()

X_train_bow = Cbow.fit_transform(X_train)
X_test_bow = Cbow.transform(X_test)

print(X_train_bow.shape)
print(X_test_bow.shape)

(2267, 17118)
(567, 17118)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vect = TfidfVectorizer()
X_train_tfidf = tfidf_vect.fit_transform(X_train)
X_test_tfidf = tfidf_vect.transform(X_test)

print(X_train_tfidf.shape)
print(X_test_tfidf.shape)


(2267, 17118)
(567, 17118)


In [ ]:
from sklearn.naive_bayes import MultinomialNB
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)
# print the multinomialNB
print(accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


0.5573192239858906
[[153   9  38]
 [ 16  83  63]
 [ 83  42  80]]
              precision    recall  f1-score   support

        High       0.61      0.77      0.68       200
         Low       0.62      0.51      0.56       162
      Medium       0.44      0.39      0.41       205

    accuracy                           0.56       567
   macro avg       0.56      0.56      0.55       567
weighted avg       0.55      0.56      0.55       567



In [33]:
# implement through gaussainNb
from sklearn.naive_bayes import GaussianNB
model = GaussianNB()
model.fit(X_train_tfidf.toarray() ,y_train)
y_pred = model.predict(X_test_tfidf.toarray())
print(accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

0.47795414462081126
[[84 35 81]
 [ 3 96 63]
 [44 70 91]]
              precision    recall  f1-score   support

        High       0.64      0.42      0.51       200
         Low       0.48      0.59      0.53       162
      Medium       0.39      0.44      0.41       205

    accuracy                           0.48       567
   macro avg       0.50      0.49      0.48       567
weighted avg       0.50      0.48      0.48       567



In [35]:
# implementing Logistic regression
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(X_train_tfidf,  y_train)
y_pred = model.predict(X_test_tfidf)
print(accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

0.5855379188712522
[[134  13  53]
 [  8 104  50]
 [ 57  54  94]]
              precision    recall  f1-score   support

        High       0.67      0.67      0.67       200
         Low       0.61      0.64      0.62       162
      Medium       0.48      0.46      0.47       205

    accuracy                           0.59       567
   macro avg       0.59      0.59      0.59       567
weighted avg       0.58      0.59      0.58       567



**Logistic Regression**
- Highest accuracy
- Highest macro F1
- Better Class Wise Performance